1. ensure the correct output with julia
2. construct the DCOPF latex tutorial
3. check the PU in the formulation, since the 30bus is infeasible

In [25]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from scipy.stats import multivariate_normal
import numpy as np
from tqdm import tqdm
file=".\\excel_outputs\\pglib_opf_case30_ieee.xlsx"
mpc_data = pd.read_excel(file, sheet_name=['baseMVA', 'bus', 'gen', 'gencost', 'branch'])
baseMVA = mpc_data['baseMVA'].values[0][-1]

In [26]:
def omega_sample(buses, Pd, sigma_scaling=0.03, nsamples=1000):
    stdomega = {b: sigma_scaling*Pd[b] for b in buses}
    # nonzeroindices = [i for i in range(len(stdomega)) if stdomega[i] > 1e-5]
    # mean = np.zeros(len(nonzeroindices))
    # cov = np.diag(list(map(stdomega.__getitem__, nonzeroindices)))**2
    mean = {b: 0 for b in buses}
    # cov = {b: stdomega[b]**2 for b in buses}
    omega_samples = {(b,s): mean[b] + stdomega[b]*np.random.randn() if stdomega[b] > 1e-5 else 0
                     for b in buses for s in range(nsamples)}
    omega_samples = {(b,s): omega_samples[b,s] if omega_samples[b,s] > 1e-5 else 0
                     for b in buses for s in range(nsamples)}
    # omega = multivariate_normal.rvs(mean=mean, cov=cov, size=nsamples)
    # omega_samples = np.zeros((len(buses), nsamples))
    # omega_samples[nonzeroindices] = omega.T if omega.ndim == 2 else omega[:, np.newaxis]
    return omega_samples

In [27]:
# Create gurobipy Model
model = gp.Model("DCOPF")
# === Sets ===
buses = mpc_data['bus']['bus_i'].tolist()
buses_index_busID = dict(zip(mpc_data['bus'].index,mpc_data['bus']['bus_i']))
buses_busID_index = dict(zip(mpc_data['bus']['bus_i'],mpc_data['bus'].index))
gens = mpc_data['gen']['gen_ID'].tolist()
branches = mpc_data['branch'].index.tolist()
branches_ftbus = dict(zip(branches,mpc_data['branch'][['bus_i', 'bus_j']].values))
# === Parameters ===
# Generator cost coefficients (all costs are incorporated)
c = {}
for i in mpc_data['gencost']["gen_ID"]:
    c[i] = [mpc_data['gencost']['c2'][i-1]*baseMVA,mpc_data['gencost']['c1'][i-1]*baseMVA,
            mpc_data['gencost']['c0'][i-1]*baseMVA]
# Bus power demand (p.u)
Pd = dict(zip(mpc_data['bus']['bus_i'], mpc_data['bus']['Pd']/baseMVA))
# shunt conductance (p.u)
Gs = dict(zip(mpc_data['bus']['bus_i'], mpc_data['bus']['Gs']/baseMVA))
# Generator capacity limits (p.u)
Pmax = dict(zip(mpc_data['gen']['gen_ID'], mpc_data['gen']['Pmax']/baseMVA))    
Pmin = dict(zip(mpc_data['gen']['gen_ID'], mpc_data['gen']['Pmin']/baseMVA))
# Transmission line limits (p.u)
Pmax_line = dict(zip(branches, mpc_data['branch']['rateA']/baseMVA))
Pmin_line = dict(zip(branches, -mpc_data['branch']['rateA']/baseMVA))
# Line susceptance (1/X), assuming per unit values
# B = dict(zip(branches, 1/(mpc_data['branch']['x'])))
B = dict(zip(branches, mpc_data['branch']['x']/(mpc_data['branch']['x']**2+mpc_data['branch']['r']**2)))
# Generator bus assignment
gen_bus = dict(zip(mpc_data['gen']['gen_ID'], mpc_data['gen']['bus_i']))
# === Variables ===
Pg = model.addVars(gens, lb=Pmin, ub=Pmax, vtype=GRB.CONTINUOUS, name="Pg")
theta = model.addVars(buses, lb=-100, ub=100, vtype=GRB.CONTINUOUS, name="theta")
P_flow = model.addVars(branches, lb=Pmin_line, ub=Pmax_line, vtype=GRB.CONTINUOUS, name="P_flow")
omega = model.addVars(buses, lb=0, ub=0, vtype=GRB.CONTINUOUS, name="omega")
# === Objective Function (Minimize Generation Cost, all costs are incorporated) ===
model.setObjective(gp.quicksum(c[i][0]*Pg[i] + c[i][1]*Pg[i] + c[i][2] for i in gens), GRB.MINIMIZE)
# === Power Balance Constraints ===
for b in buses:
    expr = (gp.quicksum(Pg[i] for i in gen_bus if gen_bus[i] == b)
            + gp.quicksum(P_flow[l] for l, ft in branches_ftbus.items() if ft[1] == b)
            - gp.quicksum(P_flow[l] for l, ft in branches_ftbus.items() if ft[0] == b))
    model.addConstr(expr == Pd[b] + Gs[b] + omega[b] , name=f"power_balance_{b}")
# === Line Flow Constraints (DC Power Flow) ===
for l in branches:
    model.addConstr(P_flow[l] == B[l]*(theta[branches_ftbus[l][0]] - theta[branches_ftbus[l][1]]), name=f"line_flow_{l}")
# === Reference Bus Constraint (Slack Bus) ===
ref_bus_index = mpc_data['bus'][mpc_data['bus']['type'] == 3].index[0]
model.addConstr(theta[buses_index_busID[ref_bus_index]] == 0, name="theta_ref") # buses_index_busID[ref_bus_index] gets the ref bus
# === Solve Model Using Gurobi ===
model.Params.OptimalityTol = 1e-8 # Higher precision
model.setParam('OutputFlag', 0) # suppress the output
model.optimize() 

Set parameter OptimalityTol to value 1e-08


In [28]:
Pmax_line

{0: 1.38,
 1: 1.52,
 2: 1.39,
 3: 1.35,
 4: 1.44,
 5: 1.39,
 6: 1.48,
 7: 1.27,
 8: 1.4,
 9: 1.48,
 10: 1.42,
 11: 0.53,
 12: 1.42,
 13: 2.67,
 14: 1.15,
 15: 2.1,
 16: 0.29,
 17: 0.29,
 18: 0.3,
 19: 0.2,
 20: 0.38,
 21: 0.29,
 22: 0.29,
 23: 0.29,
 24: 0.3,
 25: 0.33,
 26: 0.3,
 27: 0.29,
 28: 0.29,
 29: 0.29,
 30: 0.26,
 31: 0.29,
 32: 0.27,
 33: 0.25,
 34: 0.28,
 35: 0.75,
 36: 0.28,
 37: 0.28,
 38: 0.28,
 39: 1.4,
 40: 1.49}

In [22]:
class OPF_Scenarios:
    def __init__(self,noptimal, scenarios, solutions, cbases, rbases, whichbasis, whichscenario):
        self.noptimal = noptimal
        self.scenarios = scenarios
        self.solutions = solutions
        self.cbases = cbases
        self.rbases = rbases
        self.whichbasis = whichbasis
        self.whichscenario = whichscenario
        
def OPFScenarios(model, omega, omega_samples, nsamples):
    status = [None] * nsamples
    soln_p = np.zeros((nsamples, len(gens)))
    cbases = {}
    rbases = {}
    sample_omega = {}
    noptimal = 0
    for s in tqdm(range(nsamples)):
        # model.setAttr("LB", omega, omega_samples[:,s])
        # model.setAttr("UB", omega, omega_samples[:,s])
        for b in omega:
            omega[b].lb = 0
            # omega[b].lb = omega_samples[b,s]
            omega[b].ub = 0
            # omega[b].ub = omega_samples[b,s]
#         m.model.setParam('OutputFlag', 0) # suppress the output
        # print("Variables:")
        # for v in model.getVars():
        #     if (v.varName[:5] == "omega") and (v.ub!=0)and (v.lb!=0):
        #         print(f"{v.varName}: lb={v.lb}, ub={v.ub}, obj={v.Obj}")
        # if s >10:
        #     break
        model.optimize()
        status[s] = model.status
        if status[s] == GRB.OPTIMAL:
            soln_p[s,:] = list(model.getAttr('x', Pg).values())
            noptimal += 1
            cbasis = tuple(model.getAttr('Vbasis', model.getVars()))
            rbasis = tuple(model.getAttr('Cbasis', model.getConstrs()))
            cbases[cbasis] = cbases.get(cbasis, [])
            rbases[rbasis] = rbases.get(rbasis, [])
            cbases[cbasis].append(noptimal)
            rbases[rbasis].append(noptimal)
            sample_omega[s] = {b:omega_samples[b,s] for b in buses}
    assert noptimal == sum(1 for stat in status if stat == 2), 'Mismatch in optimal scenario count'
    sample_p = soln_p[np.array(status)==GRB.OPTIMAL,:]
    # sample_omega = omega_samples[:, np.array(status)==GRB.OPTIMAL]
    colbases = list(cbases.keys())
    rowbases = list(rbases.keys())
    whichcol = dict(zip(colbases, range(len(colbases))))
    whichrow = dict(zip(rowbases, range(len(rowbases))))
    whichbasis = np.zeros((noptimal, 2), dtype=int)
    for ckey in cbases.keys():
        whichbasis[cbases.get(ckey)[-1]-1,0] = whichcol[ckey]
    for rkey in rbases.keys():
        whichbasis[rbases.get(rkey)[-1]-1,1] = whichrow[rkey]
    whichscenario = {}
    for i in range(noptimal):
        basiskey = (whichbasis[i,0], whichbasis[i,1])
        whichscenario[basiskey] = whichscenario.get(basiskey, [])
        whichscenario[basiskey].append(i)
    return OPF_Scenarios(noptimal, sample_omega, sample_p, colbases, rowbases, whichbasis, whichscenario)

In [23]:
np.random.seed(0)
nsamples=5000
omega_samples = omega_sample(buses, Pd, sigma_scaling=0.03, nsamples=nsamples)
scenarios = OPFScenarios(model, omega, omega_samples,nsamples)
scenarios.noptimal

100%|██████████| 5000/5000 [00:01<00:00, 4796.19it/s]


5000

In [8]:
len(scenarios.solutions[0])

49